# Entrenamiento de Huellas-GAN en Colab

Fase 4 del proyecto: entrenamos la cDCGAN condicional en una GPU T4 de Colab Free.

**Qué va a pasar:**

1. Colab te da una máquina virtual temporal con GPU.
2. Clonamos el repo desde GitHub.
3. Subís `kaggle.json` una sola vez (para que bajemos el dataset SOCOFing).
4. Regeneramos el dataset dentro de la VM (~10 min).
5. Entrenamos la GAN (~2-2.5 h en T4 con los 150 epochs default).
6. Al final te bajás `generator.pt` y los samples a tu PC.

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)*.

## 1. Verificar GPU

In [ ]:
import torch

print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Mem total:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('OJO: no hay GPU. Entorno de ejecución -> Cambiar tipo de entorno -> GPU.')

## 2. Clonar el repo

El código se trae desde GitHub al disco de la VM.

In [ ]:
%cd /content
!rm -rf /content/huellas
!git clone https://github.com/Arielsecchi/huellas.git /content/huellas
%cd /content/huellas

## 3. Instalar dependencias

In [ ]:
!pip install -q -r requirements.txt

## 4. Subir `kaggle.json`

Te va a aparecer un botón **"Elegir archivos"**. Subí tu `kaggle.json` desde la PC (es el archivo que bajaste de kaggle.com → Settings → API → Create New Token).

In [ ]:
import os
from google.colab import files

uploaded = files.upload()
assert 'kaggle.json' in uploaded, 'Subí el archivo kaggle.json'

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'wb') as f:
    f.write(uploaded['kaggle.json'])
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('kaggle.json instalado OK')

## 5. Descargar SOCOFing + preproceso + etiquetado Vucetich

Las tres fases del pipeline de datos corren en <10 min totales. El resultado (`data/processed/images.npz` y `data/processed/metadata.csv`) queda en el disco local de la VM.

In [ ]:
!python -m src.data.download_socofing

In [ ]:
!python -m src.data.preprocess

In [ ]:
!python -m src.data.label_vucetich --full

## 6. Smoke test del training loop (2 iteraciones)

Chequea que todo enchufa antes del entrenamiento largo.

In [ ]:
from src.training.config import TrainConfig
from src.training.train import train

train(TrainConfig(epochs=1, batch_size=16, num_workers=0,
                  max_steps=2, sample_every_epochs=1, ckpt_every_epochs=1))

## 7. Entrenamiento completo

150 épocas con batch 64 tardan ~2-2.5 h en una T4 (receta: cBN en el G + hinge loss + horizontal flip con swap I↔E).

> **Importante:** no cierres la pestaña de Colab durante el entrenamiento. Si la VM se reinicia, se pierde todo lo que está en `/content/` y hay que volver a correr desde la celda 2.
>
> Si querés un primer resultado más rápido, bajá `epochs` a 80 — 50 se quedó corto en la corrida anterior pero 80 ya muestra crestas en T4 (~1 h).

In [ ]:
cfg = TrainConfig(
    epochs=150,
    batch_size=64,
    num_workers=2,
    sample_every_epochs=5,
    ckpt_every_epochs=25,
)
train(cfg)

## 8. Ver samples y curva de pérdida

In [ ]:
from IPython.display import Image, display
from pathlib import Path

samples = sorted(Path('outputs/training_samples').glob('epoch_*.png'))
print(f'{len(samples)} grillas de samples')
if samples:
    display(Image(filename=str(samples[-1])))

In [ ]:
import csv
import matplotlib.pyplot as plt

with open('outputs/training_samples/train_log.csv') as f:
    rows = list(csv.DictReader(f))
epochs = [int(r['epoch']) for r in rows]
loss_d = [float(r['loss_d']) for r in rows]
loss_g = [float(r['loss_g']) for r in rows]

plt.figure(figsize=(8, 4))
plt.plot(epochs, loss_d, label='D')
plt.plot(epochs, loss_g, label='G')
plt.xlabel('época')
plt.ylabel('loss (media de la época)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 9. Bajar resultados a tu PC

Empaquetamos el modelo final + los samples en un ZIP y lo descargamos. **Hacé esto antes de cerrar Colab**, porque cuando la VM se reinicia se borra todo.

In [ ]:
import shutil
from google.colab import files

# armamos una carpeta limpia con lo que nos llevamos
shutil.rmtree('/content/huellas_out', ignore_errors=True)
shutil.copytree('models/final', '/content/huellas_out/final')
shutil.copytree('outputs/training_samples', '/content/huellas_out/training_samples')

# zip + download
shutil.make_archive('/content/huellas_out', 'zip', '/content/huellas_out')
files.download('/content/huellas_out.zip')

## 10. Listo

El ZIP `huellas_out.zip` que se bajó a tu PC contiene:

- `final/generator.pt` — el modelo entrenado (lo usa la app web en Fase 6).
- `training_samples/epoch_XXX.png` — grilla de samples por época (muestra la evolución visual).
- `training_samples/train_log.csv` — log de pérdidas por época.

Descomprimilo dentro de la carpeta del proyecto en tu PC para seguir con Fase 5 (evaluación).